In [7]:
# %run preprocess_ruslan.py

Num rows before cleaning: 22200; after cleaning: 22200


In [2]:
%run text_filter.py


Всего ошибок предсказания: 2
Первые 10 ошибочных строк:
                                       text  is_normalized  predicted
144             Документ заверен в Минюсте.              0          1
213  Счёт за прошлый месяц уже сформирован.              0          1
F1 Score is 0.993103, Precision is 0.986301, Recall is 1.000000


In [41]:
import csv

# Имена исходного и нового файлов
input_file = 'data/dev_sentences.csv'  
output_file = 'data/only_normalized.csv'

# Открываем исходный файл для чтения, а новый — для записи
with open(input_file, mode='r', encoding='utf-8', newline='') as infile, \
     open(output_file, mode='w', encoding='utf-8', newline='') as outfile:
    
    # Настраиваем чтение и запись с разделителем |
    reader = csv.DictReader(infile, delimiter='|')
    writer = csv.DictWriter(outfile, fieldnames=reader.fieldnames, delimiter='|')
    
    # Записываем строку заголовка "text|is_normalized" в новый файл
    writer.writeheader()
    
    # Проходим по всем строкам и фильтруем их
    for row in reader:
        # Проверяем, равно ли значение в колонке is_normalized строке '0'
        if row['is_normalized'] == '1':
            writer.writerow(row)

print("Фильтрация завершена!")


Фильтрация завершена! Строки с is_normalized=0 сохранены в not_normalized.csv


In [1]:
import re

# ТЕСТОВАЯ для отладки

In [8]:
DIGIT_ROOTS = [
        "ноль", "ноля", "нуль", "нуля", "нулем", "нулём",
        "один", "одно", "одна", "одни", "одну",
        "два", "две", "двух", "двум", "двое",
        "три", "трой", "трое", "троих", "троим", "трем", "трём", "трех", "трёх",
        "четыр",
        "пять", "пяти",
        "шесть", "шести",
        "семь", "семи",
        "восем", "восьм",
        "девят", "десят",
        "надцат", "дцат", "сорок", "девянос", "сто", "ста",
        "тысяч", "миллион", "миллиард", "триллион", "квадриллион"
    ]

    # Меры - сокращения
MEASUREMENTS_LIST = [
        # Длина / Расстояние
        "мм", "см", "дм", "м", "км",
        "mm", "cm", "dm", "m", "km",
        # Масса
        "мг", "г", "кг", "т", "ц",
        # Время
        "с", "сек", "мин", "ч", "д", "дн", "сут", "нед", "дек", "мес", "г", "деслет",
        "s", "min", "h", "d", "WEE", "DAD", "MON", "a", "DEC",
        # Объём 
        "мм^3", "мм3", "куб. мм", "куб мм", "MMQ",
        "см^3", "см3", "куб. см", "куб см", "CMQ",
        "дм^3", "дм3", "куб. дм", "куб дм", "DMQ",
        "м^3", "м3", "куб. м", "куб м", "MTQ",
        "мл", "ml", "MLT", "л", "I", "L", "LTR",
        "дл", "dl", "DLT", "гл", "hl", "HLT", "Мл", "Ml", "MAL",
        "дюйм^3", "дюйм3", "куб. дюймов", "куб. дюйма", "куб дюймов", "куб дюйма", "in3", "INQ",
        "фут^3", "фут3", "куб. футов", "куб. фута", "куб футов", "куб фута", "ft3", "FTQ",
        "ярд^3", "ярд3", "куб. ярдов", "куб. ярда", "куб ярдов", "куб ярда", "yd3", "YDQ",
        # Площадь
        "мм2", "мм^2", "mm2", "mm^2", "ММК",
        "см2","см^2","cm2","cm^2","СМК",
        "дм2","дм^2","dm2","dm^2","DMК",
        "м2","м^2","m2","m^2","МТК",
        "10^3 м^2", "daa", "ТЫС М2", "ТЫС М^2", 
        "га", "ha", "HAR",
        "км2", "км^2", "km2", "km^2", "КМК", 
        "дюйм2", "дюйм^2", "in2", "in^2", "INK", 
        "фут2", "фут^2", "ft2", "ft^2", "FTK",
        "ярд2", "ярд^2", "yd2", "yd^2", "YDК",
        "Ар", "а", "a", "ARE",
        # Скорость
        "Бк", "Bq", "BQL", "вБ", "Wb", "WEB", 
        "зуз", "kn", "УЗ", "KNT",
        "м/с",  "м в сек.", "m/s", "м/сек", "MTS",
        "об/с", "об в с", "r/s", "RPS", "об в сек.", 
        "об/мин", "об в мин", "r/min", "RPM",
        "км/ч", "км в ч",  "km/h", "KMH",  
        "м/с2", "m/s2", "MSK",
        # Электричество и технич.
        "Вт", "w", "WTT", "кВт", "kW", "KWT", 
        "МВт", "MW", "МЕГАВТ", "MAW",
         "V", "VLT", "кВ", "kV", "KVT", 
        "кВ·А", "kV·A", "KVA", "МВ·А", "MV·A", "МЕГАВ·А", "MVA",
        "квар", "kVAR", "KVR", "Вт·ч", "W·h", "ВЧ·Ч", "WHR",
        "кВт·ч", "kW·h", "KWH", "МВт·ч", "MW·h", "МЕГАВТ·Ч", "MWH",
        "ГВт·ч", "GW·h", "ГИГАВТ·Ч", "GWH", 
        "А", "A", "AMP", "А·ч", "A·h", "АМН", "ТАН",
        "Кл", "С", "соu", "Дж", "J", "JOU",
        "кДж", "kJ", "KJO",
        "Ом", "Ω", "OHM", "Гр", "Gy",
        "мкГо", "μGy", "МКГР", "MKGY",
        "мГр", "mGy", "МЛГР", "MGY",
        "кГр", "kGy", "КИЛОГР", "KGY",
        "°С", "°C", "ГРАД ЦЕЛЬС", "CEL",
        "°F", "°F", "ГРАД ФАРЕНГ", "FAN",
        "кд", "cd", "CDL", "лк",  "lx", "LUX",
        "лм", "lm", "LUM", "К",  "К", "KEL",
        "Н", "N", "NEW", "Гц",  "Hz", "HTZ",
        "кГц", "kHz", "МГц", "MHz",  "МЕГАГЦ",
        "ГГц", "GHz", "ГИГАГЦ", "GHZ",
        "Па", "Pa", "PAL", "ТГц",  "THz", "ТЕРАГЦ"
    ]

measures_alternatives = '|'.join([re.escape(m) for m in MEASUREMENTS_LIST])
_digits_with_measures_pattern = re.compile(rf'\d+\s*({measures_alternatives})\b', re.IGNORECASE)

def _has_measurements_issue(text: str) -> tuple[bool, str | None]:
    """Внутренний метод для проверки правила №5.
    Возвращает (True, 'сработавшая подстрока'), если найдено нарушение, 
    иначе (False, None).
    """
    text_lower = text.lower()
    
    # Шаг А: Проверяем паттерн "цифра + мера" (например, "5 кг")
    match_digits = _digits_with_measures_pattern.search(text_lower)
    if match_digits:
        # Извлекаем оригинальный фрагмент из исходного текста (сохраняя исходный регистр)
        matched_substring = text[match_digits.start():match_digits.end()]
        return True, matched_substring
        
    # Шаг Б: Проверяем "числительное прописью + сокращение меры"
    for measure in MEASUREMENTS_LIST:
        measure_lower = measure.lower()
        
        for match in re.finditer(rf'\b{re.escape(measure_lower)}\b', text_lower):
            start_index = max(0, match.start() - 15)
            left_context = text_lower[start_index:match.start()]
            
            for root in DIGIT_ROOTS:
                if root in left_context:
                    # Находим, где именно в левом контексте начинается корень числительного
                    root_relative_idx = left_context.rfind(root)
                    # Вычисляем абсолютный индекс начала числительного в исходном тексте
                    absolute_start_idx = start_index + root_relative_idx
                    
                    # Вырезаем красивую подстроку от найденного числительного до конца меры
                    matched_substring = text[absolute_start_idx:match.end()]
                    return True, matched_substring.strip()
                    
    return False, None

has_issue, matched_text = _has_measurements_issue("Стоимость подписки — сто рублей в месяц.")
if has_issue:
    print(f"Обнаружена ошибка в тексте! Сработало на: '{matched_text}'")
